<a href="https://colab.research.google.com/github/ShreyIND/ML/blob/main/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("/content/dad_jokes.csv")

In [3]:
df

,Unnamed: 0,joke
0,0,A steak pun is a rare medium well done.
1,1,They say that breakfast is the most important ...
2,2,What do you get if you cross an angry sheep wi...
3,3,An apple a day keeps the doctor away. At least...
4,4,What sounds like a sneeze and is made of leath...
...,...,...
135594,135594,"My dog is a genius. I asked him, ""What's two m..."
135595,135595,My boss asked me why I only get sick on work d...
135596,135596,"I have a joke about chemistry, but I don’t thi..."
135597,135597,I finally watched that documentary on clocks. ...


In [4]:
df=df.iloc[:,1]

In [5]:
df

,joke
0,A steak pun is a rare medium well done.
1,They say that breakfast is the most important ...
2,What do you get if you cross an angry sheep wi...
3,An apple a day keeps the doctor away. At least...
4,What sounds like a sneeze and is made of leath...
...,...
135594,"My dog is a genius. I asked him, ""What's two m..."
135595,My boss asked me why I only get sick on work d...
135596,"I have a joke about chemistry, but I don’t thi..."
135597,I finally watched that documentary on clocks. ...


In [6]:
l=list(df)

In [7]:
l=l[:5000]

In [8]:
import tensorflow


In [9]:
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [10]:
from tensorflow.keras.models import Sequential

In [11]:
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense,LSTM

In [12]:
model=Sequential()

In [13]:
tokenizer=Tokenizer()

In [14]:
tokenizer.fit_on_texts(l)

In [15]:
input_sequences = []

for line in l:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for k in range(1, len(token_list)):
        n_gram_sequence = token_list[:k+1]
        input_sequences.append(n_gram_sequence)

print("First 5 sequences:", input_sequences[:5])

First 5 sequences: [[1, 665], [1, 665, 913], [1, 665, 913, 16], [1, 665, 913, 16, 1], [1, 665, 913, 16, 1, 914]]


In [16]:
max_len=0
for i in input_sequences:
    if max_len<len(i):
        max_len=len(i)

In [17]:
input_sequences=np.array(pad_sequences(input_sequences,maxlen=max_len,padding='pre'))

In [18]:
x=input_sequences[:,:-1]
y=input_sequences[:,-1]

In [19]:
y=tensorflow.keras.utils.to_categorical(y,num_classes=len(tokenizer.word_index)+1)
from tensorflow.keras.layers import Embedding
model.add(Embedding(len(tokenizer.word_index)+1, 100, input_length=max_len))
model.add(LSTM(150))
model.add(Dense(len(tokenizer.word_index)+1, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [20]:
model.compile(loss='categorical_crossentropy', optimizer='adam',metrics=['accuracy'])

In [21]:
model.fit(x,y,epochs=10,verbose=1)

Epoch 1/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 30s 10ms/step - accuracy: 0.0553 - loss: 6.8728
Epoch 2/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 38s 10ms/step - accuracy: 0.1527 - loss: 5.3669
Epoch 3/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.2567 - loss: 4.3326
Epoch 4/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.3694 - loss: 3.4975
Epoch 5/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.4646 - loss: 2.8719
Epoch 6/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.5351 - loss: 2.4046
Epoch 7/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.6007 - loss: 2.0303
Epoch 8/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.6515 - loss: 1.7297
Epoch 9/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.6960 - loss: 1.4943
Epoch 10/10
2430/2430 ━━━━━━━━━━━━━━━━━━━━ 24s 10ms/step - accuracy: 0.7294 - loss: 1.3118


In [22]:
def complete_joke(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted_probs, axis=-1)[0]
        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_word_index:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

# --- 6. Test with some Dad Joke starts ---
print("\n--- Generated Dad Jokes ---")
print(complete_joke("Why did the", 5))
print(complete_joke("My wife", 6))
print(complete_joke("I went to", 5))


--- Generated Dad Jokes ---
Why did the scarecrow win an award because
My wife left a note on the fridge
I went to buy a clock i was
